# Sensor placement optimization + Isaac Lab (Colab)

This notebook is the **practical** Colab entrypoint for this repository when you use the unofficial Isaac Sim / Isaac Lab Colab installer workflow (same idea as `isaac_lab_2_1_0_colab.ipynb` in this folder).

**What you get end-to-end**
- Installs Isaac Sim + Isaac Lab (same script pattern as the uploaded Colab)
- Clones *this* repo to `/content/sensor_placement_opt`
- Launches a small **HTTP JSON bridge** implemented in:
  - `scripts/isaaclab_sensor_bridge.py`
- Runs the outer-loop optimizer in `sensor_opt` against `IsaacSimEvaluator` using `http://127.0.0.1:<port>`

**What you must customize for real research**
- In `scripts/isaaclab_sensor_bridge.py`, implement:
  - `_apply_sensor_config(...)` to actually move/enable sensors in your USD scene
  - meaningful metric mapping in `_rollout_one_env(...)` (right now it includes a *placeholder* success proxy for CartPole)


## Configure your 3D LiDAR + RGB-D mounts in the USD stage (recommended)

`SensorConfig` already describes each sensor (type, mounting slot, offsets, FoV fractions). To actually *move* sensors in Isaac, set USD prim paths in `configs/default.yaml` under:
- `sensor_models.lidar.isaac.prim_path`
- `sensor_models.camera.isaac.prim_path`

For parallel env cloning, paths often look like:
`/World/envs/env_{env_idx}/Robot/.../YourLidar`
`/World/envs/env_{env_idx}/Robot/.../YourDepthCamera`

The bridge will `.format(env_idx=...)` if `{env_idx}` is present.

## Perception metrics (what gets optimized)

`--bridge-mode ground` tries to compute `blind_spot_fraction` from **depth** + **3D point-like** arrays found inside Isaac Lab observations (task-dependent keys). If it cannot find them, it falls back to the repo’s analytic `fast_baseline_metrics(...)` for that episode and prints a one-time warning.


In [ ]:
# Install Isaac Sim 4.5 and Isaac Lab 2.1.0. This process takes about 10 mins to complete.
!wget -O install-isaac-sim-4.5.sh https://raw.githubusercontent.com/j3soon/isaac-sim-colab/main/scripts/install-isaac-sim-4.5.sh
!time bash install-isaac-sim-4.5.sh
!wget -O install-isaac-lab-2.1.0.sh https://raw.githubusercontent.com/j3soon/isaac-sim-colab/main/scripts/install-isaac-lab-2.1.0.sh
!time bash install-isaac-lab-2.1.0.sh


In [ ]:
# Clone *this* repository (the sensor placement optimizer) and install its Python deps.
# NOTE: if you are iterating on a fork, change REPO_URL.

import os
import subprocess
import sys

REPO_URL = "https://github.com/dcaglar-28/sensor_placement_opt.git"
REPO_DIR = "/content/sensor_placement_opt"

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    # IMPORTANT: in Colab, use `{REPO_DIR}` (Python formatting), not `"$REPO_DIR"` in `!` shells.
    !rm -rf "{REPO_DIR}"
    !git clone "{REPO_URL}" "{REPO_DIR}"

os.chdir(REPO_DIR)

# Install into the *same interpreter as this kernel* (avoids "pip installed but import fails")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", "pip"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"])

print("OK:", os.getcwd())
print("python:", sys.executable)


In [ ]:
# --- Kernel dependency sanity check (run right after `pip install -r requirements.txt`) ---

import importlib
import sys

for mod in ("cma", "yaml", "numpy"):
    importlib.import_module(mod)

import cma

print("OK: imports work")
print("python:", sys.executable)
print("cma:", cma.__version__)


## Start the Isaac Lab HTTP bridge (background process)

`scripts/isaaclab_sensor_bridge.py` follows the Isaac Lab pattern: `AppLauncher` → `parse_env_cfg` → `gym.make` → `env.step` (Isaac Lab’s `scripts/environments/random_agent.py` is the canonical example).

**Colab note:** the bridge should run as a **background** process (via `nohup`) so the next notebook cell can run the optimizer in the *same* kernel session.

Set `PORT` to match the optimizer cell.

If you get import errors, re-run the Isaac install cell, and confirm you are using the same Python environment the installer configured (typically `/usr/local/bin/python` in these Colab recipes).


In [ ]:
import os
import json
import time
import urllib.error
import urllib.request

# Same directory as the clone cell above
REPO_DIR = "/content/sensor_placement_opt"
os.chdir(REPO_DIR)

# Isaac Sim / Kit environment flags (per Isaac Sim install docs)
os.environ["OMNI_KIT_ACCEPT_EULA"] = "YES"

PORT = 8010
# Default Isaac Lab task (change if your install/registry differs)
# - `Isaac-Velocity-Flat-Unitree-Go2-v0` is a common Colab/Isaac Lab smoke task (legged locomotion).
# - If you want ANYmal navigation instead, set TASK manually after printing available env IDs.
TASK = os.environ.get("ISAAC_TASK", "Isaac-Velocity-Flat-Unitree-Go2-v0")
NUM_ENVS = 1

BRIDGE = f"{REPO_DIR}/scripts/isaaclab_sensor_bridge.py"
LOG = "/tmp/isaaclab_sensor_bridge.log"

!pkill -f "isaaclab_sensor_bridge.py" 2>/dev/null || true
# Use the same `python` as this kernel for predictable imports/paths
!nohup python "{BRIDGE}" --host 127.0.0.1 --port {PORT} --bridge-mode ground --headless --enable_cameras --task "{TASK}" --num_envs {NUM_ENVS} --repo-root "{REPO_DIR}" > "{LOG}" 2>&1 &

# The bridge only binds HTTP after `AppLauncher` + `gym.make` init; that can take many minutes in Colab.
# Poll `/health` instead of a fixed `sleep 2`.
deadline = time.time() + 60 * 30  # 30 minutes
last_log_pos = 0

def _read_new_log_bytes() -> bytes:
    global last_log_pos
    try:
        with open(LOG, "rb") as f:
            f.seek(last_log_pos)
            chunk = f.read()
            last_log_pos = f.tell()
            return chunk
    except FileNotFoundError:
        return b""

print("Waiting for HTTP bridge to become ready on 127.0.0.1:%d/health ..." % PORT, flush=True)
while time.time() < deadline:
    # show incremental logs (helpful while Isaac is starting)
    chunk = _read_new_log_bytes()
    if chunk:
        print(chunk.decode("utf-8", errors="replace"), end="")

    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{PORT}/health", timeout=2) as r:
            body = r.read()
        print("Bridge is ready:", body.decode("utf-8", errors="replace").strip(), flush=True)
        break
    except (urllib.error.URLError, TimeoutError, ConnectionError, OSError):
        time.sleep(5)
else:
    raise RuntimeError("Timed out waiting for bridge /health. See log: " + LOG)

# show last part of the log for manual debugging
!tail -n 60 "{LOG}" || true
print("Bridge log:", LOG)


## Run the sensor placement optimizer against the local bridge

This uses the same JSON client contract as `notebooks/sensor_placement_opt_colab.ipynb`.

It runs a **tiny** CMA smoke test by mutating `configs/default.yaml` in-memory (1 generation, small population).


In [ ]:
import copy
import json
import os
import urllib.request
from dataclasses import asdict

import numpy as np
import yaml

from sensor_opt.encoding.config import SensorConfig
from sensor_opt.inner_loop.isaac_evaluator import IsaacSimEvaluator
from sensor_opt.logging.experiment_logger import ExperimentLogger
from sensor_opt.loss.loss import EvalMetrics
from sensor_opt.search.factory import create_search


class IsaacRpcEnvClient:
    """Minimal JSON-over-HTTP client matching `IsaacSimEvaluator`'s env contract."""

    def __init__(self, base_url: str, timeout_sec: float = 600.0):
        self.base_url = base_url.rstrip("/")
        self.timeout_sec = float(timeout_sec)

    def reconfigure_sensors(self, env_idx: int, config: SensorConfig, sensor_models: dict) -> None:
        payload = {"env_idx": int(env_idx), "config": asdict(config), "sensor_models": sensor_models}
        self._post_json("/reconfigure_sensors", payload)

    def run_rollouts(self, n_episodes: int, rng: np.random.Generator) -> list[EvalMetrics]:
        seed = int(rng.integers(0, 2**31 - 1))
        payload = {"n_episodes": int(n_episodes), "seed": int(seed)}
        data = self._post_json("/run_rollouts", payload)
        return [
            EvalMetrics(
                collision_rate=float(m["collision_rate"]),
                blind_spot_fraction=float(m["blind_spot_fraction"]),
                mean_goal_success=float(m["mean_goal_success"]),
                n_episodes=int(m["n_episodes"]),
            )
            for m in data["metrics"]
        ]

    def _post_json(self, path: str, payload: dict):
        url = f"{self.base_url}{path}"
        req = urllib.request.Request(
            url,
            data=json.dumps(payload).encode("utf-8"),
            headers={"Content-Type": "application/json"},
            method="POST",
        )
        with urllib.request.urlopen(req, timeout=self.timeout_sec) as resp:
            return json.loads(resp.read().decode("utf-8"))


os.chdir("/content/sensor_placement_opt")

PORT = 8010  # must match the bridge cell
NUM_ENVS = 1  # must match --num_envs in the bridge cell

env = IsaacRpcEnvClient(base_url=f"http://127.0.0.1:{PORT}", timeout_sec=60 * 30)
base_evaluator = IsaacSimEvaluator(isaac_sim_cfg={"env": env, "num_envs": NUM_ENVS})

cfg = yaml.safe_load(open("configs/default.yaml"))
cfg = copy.deepcopy(cfg)
cfg["inner_loop"]["mode"] = "isaac_sim"
cfg["multi_fidelity"]["enabled"] = False

# tiny smoke settings
cfg["cma"]["max_generations"] = 1
cfg["cma"]["population_size"] = 4
cfg["cma"]["tolx"] = 1e-9
cfg["cma"]["tolfun"] = 1e-9

seed = int(cfg.get("experiment", {}).get("seed", 42))

with ExperimentLogger(
    experiment_name=cfg["experiment"]["name"] + "_isaaclab_smoke",
    results_dir=cfg.get("logging", {}).get("results_dir", "results"),
    use_mlflow=False,
    mlflow_tracking_uri=cfg.get("logging", {}).get("mlflow_tracking_uri", "mlruns"),
    run_config=cfg,
) as logger:
    search = create_search(
        search_type=cfg["search"]["type"],
        config=cfg,
        evaluator={
            "evaluator_fn": None,
            "evaluator": None,
            "base_evaluator": base_evaluator,
            "logger": logger,
            "seed": seed,
        },
    )
    result = search.run()

print("best_loss:", result.best_loss)
print("run_id:", result.run_id)
